In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error, median_absolute_error

#load in the cleaned test and training datasets
test_df = pd.read_csv("cleaned_test.csv")
train_df = pd.read_csv("cleaned_training.csv")

feature_cols = [
    'LivingArea', 
    'BedroomsTotal', 
    'BathroomsTotalInteger',
    'LotSizeSquareFeet', 
    'zip_median_price', 
    'city_median_price',
    'bed_bath_ratio', 
    'property_age', 
    'district_median_price'
]

X_train = train_df[feature_cols].copy()
y_train = train_df['ClosePrice'].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df['ClosePrice'].copy()

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")

X_train shape: (71099, 9) | y_train shape: (71099,)
X_test shape:  (12784, 9)  | y_test shape:  (12784,)


Baseline XGBoost Model

In [20]:
# Start with default XGBoost to get a baseline
xgb_default = XGBRegressor(random_state=42)
xgb_default.fit(X_train, y_train)

y_train_pred = xgb_default.predict(X_train)
y_test_pred = xgb_default.predict(X_test)

xgb_train_results = r2_score(y_train, y_train_pred)
xgb_test_results = r2_score(y_test, y_test_pred)
print('Baseline XGBoost Results: \n')
print(f'Training R²: {xgb_train_results:.6f}')
print(f'Test R²: {xgb_test_results:.6f}')

Baseline XGBoost Results: 

Training R²: 0.932133
Test R²: 0.868322


Hyperparameter tuned models

In [ ]:
#n_estimators: #of trees; higher values -> more complex model, lower values -> simpler model
#learning rate: higher values -> faster learning, lower values -> slower learning
#max_depth: higher values -> capture complex relationships, lower values -> simpler relationships
xgb_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.10, 0.05, 0.03],
    'max_depth': [3, 6, 9]
}

results = []

for n_estimators in xgb_params['n_estimators']:
    for learning_rate in xgb_params['learning_rate']:
        for max_depth in xgb_params['max_depth']:
            
            # Train the model
            model = XGBRegressor(
                n_estimators=n_estimators,
                learning_rate=learning_rate,
                max_depth=max_depth,
                random_state=42,
                objective='reg:squarederror'
            )
            model.fit(X_train, y_train)
            
            # Predict
            y_train_pred = model.predict(X_train)
            y_test_pred = model.predict(X_test)

            xgb_train_r2 = r2_score(y_train, y_train_pred)
            xgb_test_r2 = r2_score(y_test, y_test_pred)
            
            # Store parameters and scores
            results.append({
                'n_estimators': n_estimators,
                'learning_rate': learning_rate,
                'max_depth': max_depth,
                'Training R²': round(xgb_train_r2, 6),
                'Test R²': round(xgb_test_r2, 6)
            })

# Sort by best Test R² score
results_df = pd.DataFrame(results).sort_values(by='Test R²', ascending=False).reset_index(drop=True)

# Display top 5 best parameter combinations
display(results_df.head(5))

best_model = results_df.iloc[0]

print("\n Best Model Parameters:")
print(f"n_estimators:  {best_model['n_estimators']}")
print(f"learning_rate: {best_model['learning_rate']}")
print(f"max_depth:     {best_model['max_depth']}")
print(f"Training R²:   {best_model['Training R²']}")
print(f"Test R²:       {best_model['Test R²']}")

,n_estimators,learning_rate,max_depth,Training R²,Test R²
0,300,0.10,9,0.978647,0.872436
1,200,0.10,9,0.970056,0.872362
2,300,0.05,9,0.962805,0.872064
3,300,0.10,6,0.930471,0.871649
4,100,0.10,9,0.953461,0.871491



 Best Model Parameters:
n_estimators:  300.0
learning_rate: 0.1
max_depth:     9.0
Training R²:   0.978647
Test R²:       0.872436


In [22]:
comparison_df = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
        "BaselineXGBoost",
        "TunedXGBoost"
    ],
    "Test R²": [
        0.761132,
        0.710998,
        0.836335,
        0.868322,
        0.872436
    ]
})

display(comparison_df)

,Model,Test R²
0,Linear Regression,0.761132
1,Decision Tree,0.710998
2,Random Forest,0.836335
3,BaselineXGBoost,0.868322
4,TunedXGBoost,0.872436
